# 📘 스코프와 클로저

변수가 **어디에서 접근 가능한지**를 결정하는 규칙이 스코프이고,
함수가 **자신이 생성될 때의 변수를 기억**하는 것이 클로저입니다.

**학습 목표:**
- LEGB 규칙 (Local → Enclosing → Global → Built-in)
- global과 nonlocal 키워드
- 클로저와 팩토리 패턴

## 1. LEGB 스코프 규칙

파이썬은 변수를 찾을 때 **L → E → G → B** 순서로 검색합니다.

| 영역 | 설명 | 예시 |
|------|------|------|
| **L**ocal | 함수 내부 | `def f(): x = 1` |
| **E**nclosing | 중첩 함수의 외부 | `def outer(): x = 1; def inner(): ...` |
| **G**lobal | 모듈 전체 | 파일 최상단 `x = 1` |
| **B**uilt-in | 파이썬 내장 | `len`, `print`, `range` |

In [ ]:
# ┌───────────────────────────────────────────────┐
# │  LEGB 규칙: 변수 검색 순서                        │
# │  Local → Enclosing → Global → Built-in            │
# │  가장 가까운 스코프에서 먼저 찾습니다               │
# └───────────────────────────────────────────────┘

# Local scope: 함수 내부에서만 접근 가능
def my_function():
    local_var = 10
    print(f"지역 변수: {local_var}")

my_function()


In [ ]:
# print(local_var)  # NameError! 함수 밖에서 접근 불가

# Global scope: 모듈 전체에서 접근 가능
global_var = 100

def read_global():
    print(f"전역 변수 읽기: {global_var}")   # 읽기는 OK

read_global()


In [ ]:
# ┌─────────────────────────────────────────┐
# │  주의: 함수 내부에서 대입하면                │
# │  새로운 지역 변수가 생성됩니다!              │
# │  전역 변수를 수정하려면 global 키워드 필요    │
# └─────────────────────────────────────────┘

def modify_global_wrong():
    global_var = 200    # 새로운 지역 변수 생성!
    print(f"함수 내부: {global_var}")   # 200

modify_global_wrong()
print(f"함수 외부: {global_var}")   # 100 (변경 안 됨!)


In [ ]:
# global 키워드로 전역 변수 수정
def modify_global_correct():
    global global_var
    global_var = 300    # 전역 변수 수정

modify_global_correct()
print(f"global 수정 후: {global_var}")   # 300


## 2. nonlocal과 중첩 함수

중첩 함수에서 **외부 함수의 변수**를 수정하려면 `nonlocal` 키워드가 필요합니다.

In [ ]:
# Enclosing scope: 중첩 함수
def outer():
    x = "외부 변수"
    
    def inner():
        print(f"내부에서 읽기: {x}")   # 외부 함수의 변수 읽기 OK
    
    inner()
    print(f"외부에서 읽기: {x}")

outer()


In [ ]:
# ┌─────────────────────────────────────────┐
# │  nonlocal 키워드                           │
# │  중첩 함수에서 외부 함수의 변수를 수정할 때   │
# │  global과 비슷하지만 전역이 아닌 외부 스코프  │
# └─────────────────────────────────────────┘


In [ ]:
def counter():
    count = 0
    
    def increment():
        nonlocal count    # 외부 함수의 count를 가리킴
        count += 1
        return count
    
    def decrement():
        nonlocal count
        count -= 1
        return count
    
    print(f"increment: {increment()}")   # 1
    print(f"increment: {increment()}")   # 2
    print(f"decrement: {decrement()}")   # 1
    return count

result = counter()
print(f"최종 count: {result}")    # 1


## 3. 클로저(Closure)

클로저는 **함수가 자신이 생성될 때의 환경(변수)을 기억**하는 기능입니다.
함수를 반환하는 함수(팩토리 패턴)에서 핵심 역할을 합니다.

In [ ]:
# ┌───────────────────────────────────────────────────┐
# │  클로저(Closure)                                     │
# │  함수가 정의될 때의 스코프에 있는 변수를 "기억"하는 것   │
# │  외부 함수가 종료된 후에도 변수에 접근 가능             │
# └───────────────────────────────────────────────────┘

# 팩토리 패턴: 함수를 생성하는 함수
def make_multiplier(factor):
    """factor를 기억하는 곱셈 함수를 반환"""
    def multiply(x):
        return x * factor    # 외부 함수의 factor를 기억!
    return multiply

double = make_multiplier(2)    # factor=2를 기억
triple = make_multiplier(3)    # factor=3을 기억

print(f"double(5) = {double(5)}")   # 10
print(f"triple(5) = {triple(5)}")   # 15
print(f"make_multiplier(10)(4) = {make_multiplier(10)(4)}")  # 40


In [ ]:
# 클로저로 상태 유지
def make_counter(start=0):
    """시작값을 기억하는 카운터"""
    count = [start]    # 리스트로 감싸서 변경 가능하게
    
    def increment():
        count[0] += 1
        return count[0]
    
    def decrement():
        count[0] -= 1
        return count[0]


In [ ]:
    def get_count():
        return count[0]
    
    return increment, decrement, get_count

inc, dec, get = make_counter(10)
print(f"초기값: {get()}")      # 10
print(f"증가: {inc()}")         # 11
print(f"증가: {inc()}")         # 12
print(f"감소: {dec()}")         # 11
print(f"현재값: {get()}")      # 11


In [ ]:
# 클로저로 권한 검사
def make_authenticator(username, password):
    """로그인 정보를 기억하는 인증기"""
    def authenticate(input_user, input_pass):
        if input_user == username and input_pass == password:
            return True
        return False
    return authenticate

auth = make_authenticator("admin", "secret123")
print(f"\n올바른 로그인: {auth('admin', 'secret123')}")   # True
print(f"잘못된 로그인: {auth('admin', 'wrong')}")         # False


## 🎯 연습 문제

1. `global` 키워드를 사용해 함수 내부에서 전역 변수 `counter`를 1씩 증가시키는 `increment()` 함수를 작성하세요.
2. 클로저를 사용해 `make_adder(n)`을 호출하면 `n`을 더하는 함수를 반환하도록 작성하세요. (`add5 = make_adder(5)`, `add5(10)` → 15)
3. 클로저를 사용해 `make_account(initial)`을 호출하면 `deposit(amount)`, `withdraw(amount)`, `get_balance()` 세 함수를 반환하도록 작성하세요.

In [ ]:
# 연습 문제 풀이
